In [1]:
import cv2
import os
import numpy as np

input_root = "segmented_frames"
output_root = "tracked_segmented_videos"

os.makedirs(output_root, exist_ok=True)

for match in sorted(os.listdir(input_root)):

    match_input = os.path.join(input_root, match)

    if not os.path.isdir(match_input):
        continue

    frames = sorted(os.listdir(match_input))

    if len(frames) == 0:
        continue

    # INIT VIDEO WRITER
    first_frame = cv2.imread(os.path.join(match_input, frames[0]))
    h, w, _ = first_frame.shape

    video_path = os.path.join(output_root, f"{match}.mp4")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(video_path, fourcc, 30, (w, h))

    print("Processing:", match)

    object_centroids = {}
    object_id = 0

    # FRAME LOOP
    for file in frames:

        frame_path = os.path.join(match_input, file)
        frame = cv2.imread(frame_path)

        if frame is None:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY)

        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        new_centroids = {}

        for c in contours:

            if cv2.contourArea(c) < 300:
                continue

            x, y, w_box, h_box = cv2.boundingRect(c)

            cx = int(x + w_box/2)
            cy = int(y + h_box/2)

            matched_id = None
            min_dist = 9999

            for obj_id, (px, py) in object_centroids.items():

                dist = np.sqrt((cx - px)**2 + (cy - py)**2)

                if dist < 50 and dist < min_dist:
                    min_dist = dist
                    matched_id = obj_id

            if matched_id is None:
                matched_id = object_id
                object_id += 1

            new_centroids[matched_id] = (cx, cy)

            # draw box
            cv2.rectangle(frame, (x, y), (x+w_box, y+h_box), (0,255,0), 2)
            cv2.putText(frame,
                        f"Player {matched_id}",
                        (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0,255,0),
                        2)

        object_centroids = new_centroids

        # ONLY VIDEO WRITE (NO SAVE IMAGES)
        out.write(frame)

    out.release()

print("Video generation completed!")

Processing: .ipynb_checkpoints
Processing: match1
Processing: match2
Processing: match3
Video generation completed!
